STAGE 8 — COST-SENSITIVE FRAUD DETECTION

The idea is different from Stage 7.

Stage 7 asked:

Which threshold gives the best statistical trade-off?

Stage 8 asks:

What happens if missing a fraud is much more expensive than investigating a genuine transaction?

We'll assign:

False Positive = investigation cost
False Negative = missed-fraud cost

Then calculate:

Cost=FP×C
FP
	​

+FN×C
FN
	​


In [87]:
# ================================================================
# STAGE 8 — COST-SENSITIVE FRAUD DETECTION
# ================================================================

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

print("=" * 70)
print("STAGE 8 — COST-SENSITIVE FRAUD DETECTION")
print("=" * 70)

# ------------------------------------------------
# Use validation probabilities from baseline XGBoost
# ------------------------------------------------

val_proba = xgb_baseline.predict_proba(X_val)[:, 1]

# ------------------------------------------------
# Define cost scenarios
# ------------------------------------------------

cost_scenarios = {
    "Balanced": {
        "FP": 1,
        "FN": 5
    },
    "Fraud-sensitive": {
        "FP": 1,
        "FN": 10
    },
    "Highly-fraud-sensitive": {
        "FP": 1,
        "FN": 20
    }
}

thresholds = np.arange(0.01, 1.00, 0.01)

results = []

for scenario, costs in cost_scenarios.items():

    for threshold in thresholds:

        pred = (val_proba >= threshold).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_val,
            pred,
            labels=[0, 1]
        ).ravel()

        cost = (
            fp * costs["FP"] +
            fn * costs["FN"]
        )

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0

        results.append({
            "Scenario": scenario,
            "Threshold": threshold,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "TN": tn,
            "Precision": precision,
            "Recall": recall,
            "Cost": cost
        })

cost_df = pd.DataFrame(results)

# ------------------------------------------------
# Best threshold for every cost scenario
# ------------------------------------------------

for scenario in cost_scenarios:

    subset = cost_df[
        cost_df["Scenario"] == scenario
    ]

    best = subset.loc[
        subset["Cost"].idxmin()
    ]

    print("\n" + "=" * 70)
    print(f"SCENARIO: {scenario}")
    print("=" * 70)

    print(f"FP cost : {cost_scenarios[scenario]['FP']}")
    print(f"FN cost : {cost_scenarios[scenario]['FN']}")

    print(f"\nOptimal threshold : {best['Threshold']:.2f}")
    print(f"TP                 : {int(best['TP'])}")
    print(f"FP                 : {int(best['FP'])}")
    print(f"FN                 : {int(best['FN'])}")
    print(f"TN                 : {int(best['TN'])}")
    print(f"Precision          : {best['Precision']:.4f}")
    print(f"Recall             : {best['Recall']:.4f}")
    print(f"Total cost         : {best['Cost']:.2f}")

# ------------------------------------------------
# Save
# ------------------------------------------------

cost_df.to_csv(
    "stage8_cost_sensitive_results.csv",
    index=False
)

print("\nSaved: stage8_cost_sensitive_results.csv")

STAGE 8 — COST-SENSITIVE FRAUD DETECTION

SCENARIO: Balanced
FP cost : 1
FN cost : 5

Optimal threshold : 0.18
TP                 : 36
FP                 : 1
FN                 : 12
TN                 : 28324
Precision          : 0.9730
Recall             : 0.7500
Total cost         : 61.00

SCENARIO: Fraud-sensitive
FP cost : 1
FN cost : 10

Optimal threshold : 0.01
TP                 : 39
FP                 : 19
FN                 : 9
TN                 : 28306
Precision          : 0.6724
Recall             : 0.8125
Total cost         : 109.00

SCENARIO: Highly-fraud-sensitive
FP cost : 1
FN cost : 20

Optimal threshold : 0.01
TP                 : 39
FP                 : 19
FN                 : 9
TN                 : 28306
Precision          : 0.6724
Recall             : 0.8125
Total cost         : 199.00

Saved: stage8_cost_sensitive_results.csv
